# 03 — Gold (SCD2) — Star Schema

Camada Gold com modelo dimensional:
- **Dynamic Tables** com refresh automático a cada 1 hora
- **SCD Tipo 2** — `_DATA_INCLUSAO` e `_DATA_ALTERACAO` para rastrear mudanças históricas
- **Fatos:** FCT_GAME (por jogo) + FCT_SHOT (por arremesso)
- **Dimensões:** DIM_SEASON + DIM_OPPONENT + DIM_ARENA

### Fontes
| Gold | Silver source |
|------|---------------|
| FCT_GAME | KOBE_GAMELOG + KOBE_POINTS_BRONZE + ARENAS |
| FCT_SHOT | SHOTS_ENRICHED |
| DIM_SEASON | KOBE_SEASONS + SHOTS_ENRICHED (agregado) |
| DIM_OPPONENT | KOBE_GAMELOG (agregado) |
| DIM_ARENA | ARENAS |

*Co-authored with CoCo*

In [ ]:
%%sql -r ctx_gold
USE WAREHOUSE YDS_WH;
USE SCHEMA YDS_DB.GOLD;

In [ ]:
%%sql -r drop_gold
-- Limpar objetos existentes
DROP DYNAMIC TABLE IF EXISTS YDS_DB.GOLD.FCT_GAME;
DROP DYNAMIC TABLE IF EXISTS YDS_DB.GOLD.FCT_SHOT;
DROP DYNAMIC TABLE IF EXISTS YDS_DB.GOLD.DIM_SEASON;
DROP DYNAMIC TABLE IF EXISTS YDS_DB.GOLD.DIM_OPPONENT;
DROP DYNAMIC TABLE IF EXISTS YDS_DB.GOLD.DIM_ARENA;

## Tabelas Fato

In [ ]:
%%sql -r create_fct_game
-- FCT_GAME: 1 linha = 1 jogo (stats oficiais + arena temporal)
-- SCD2: _DATA_INCLUSAO + _DATA_ALTERACAO
CREATE OR REPLACE DYNAMIC TABLE YDS_DB.GOLD.FCT_GAME
    TARGET_LAG = '1 hour'
    WAREHOUSE = YDS_WH
AS
SELECT
    g.GAME_ID,
    g.DATE,
    g.SEASON,
    g.GAME_TYPE,
    g.GAME_TYPE_FULL,
    g.ADVERSARIO_SIGLA,
    g.ADVERSARIO_NOME,
    g.LOCAL_JOGO,
    g.LOCAL_JOGO_DESC,
    -- Arena (join temporal)
    CASE 
        WHEN g.LOCAL_JOGO = 'Home' THEN lal.ARENA_NAME
        ELSE opp.ARENA_NAME
    END AS ARENA_NAME,
    CASE 
        WHEN g.LOCAL_JOGO = 'Home' THEN lal.CITY
        ELSE opp.CITY
    END AS CITY,
    -- Pontuação oficial (KOBE_POINTS_BRONZE = fonte de verdade)
    p.TOTAL_POINTS AS KOBE_POINTS,
    p.FG2_MADE,
    p.FG3_MADE,
    p.FG_MADE,
    p.FG_ATT,
    p.FG_PCT,
    p.FT_MADE,
    p.FT_ATT,
    p.FT_PCT,
    -- Placar e resultado
    g.LAL_SCORE,
    g.OPP_SCORE,
    g.POINT_DIFF,
    g.RESULT,
    g.IS_WIN,
    g.IS_BLOWOUT,
    g.IS_CLOSE_GAME,
    -- Auditoria SCD2
    CURRENT_TIMESTAMP() AS _DATA_INCLUSAO,
    CURRENT_TIMESTAMP() AS _DATA_ALTERACAO
FROM YDS_DB.SILVER.KOBE_GAMELOG g
LEFT JOIN YDS_DB.RAW.KOBE_POINTS_BRONZE p ON g.GAME_ID = p.GAME_ID
LEFT JOIN YDS_DB.SILVER.ARENAS lal 
    ON lal.TEAM_ABBR = 'LAL' AND g.DATE BETWEEN lal.VALID_FROM AND lal.VALID_TO
LEFT JOIN YDS_DB.SILVER.ARENAS opp 
    ON opp.TEAM_ABBR = g.ADVERSARIO_SIGLA AND g.DATE BETWEEN opp.VALID_FROM AND opp.VALID_TO;

In [ ]:
%%sql -r create_fct_shot
-- FCT_SHOT: 1 linha = 1 arremesso (dados enriquecidos)
-- SCD2: _DATA_INCLUSAO + _DATA_ALTERACAO
CREATE OR REPLACE DYNAMIC TABLE YDS_DB.GOLD.FCT_SHOT
    TARGET_LAG = '1 hour'
    WAREHOUSE = YDS_WH
AS
SELECT
    MATCH_EVENT_ID,
    MATCH_ID AS GAME_ID,
    SHOT_ID_NUMBER,
    DATE_OF_GAME,
    GAME_SEASON AS SEASON,
    ADVERSARIO,
    GAME_LOCATION,
    LOCATION_X,
    LOCATION_Y,
    AREA_OF_SHOT,
    SHOT_BASICS,
    RANGE_OF_SHOT,
    DISTANCE_OF_SHOT,
    SHOT_ZONE_CUSTOM,
    PERIODO,
    IS_PLAYOFF,
    TYPE_OF_SHOT,
    SHOT_TYPE_FINAL,
    SHOT_CATEGORY,
    IS_GOAL,
    SHOT_RESULT,
    IS_CLUTCH,
    IS_BACKCOURT_SHOT,
    IS_RECOVERED,
    REMAINING_MIN,
    REMAINING_SEC,
    TOTAL_REMAINING_SEC,
    -- Auditoria SCD2
    CURRENT_TIMESTAMP() AS _DATA_INCLUSAO,
    CURRENT_TIMESTAMP() AS _DATA_ALTERACAO
FROM YDS_DB.SILVER.SHOTS_ENRICHED;

## Tabelas Dimensão

In [ ]:
%%sql -r create_dim_season
-- DIM_SEASON: 1 linha = 1 temporada com métricas agregadas
CREATE OR REPLACE DYNAMIC TABLE YDS_DB.GOLD.DIM_SEASON
    TARGET_LAG = '1 hour'
    WAREHOUSE = YDS_WH
AS
SELECT
    s.SEASON,
    s.GAMES,
    s.PPG,
    s.WINS,
    s.LOSSES,
    s.TOTAL_GAMES_TEAM,
    s.WIN_PCT,
    COUNT(sh.IS_GOAL) AS TOTAL_SHOTS_TRACKED,
    SUM(CASE WHEN sh.IS_GOAL = 1 THEN 1 ELSE 0 END) AS TOTAL_MADE_TRACKED,
    ROUND(SUM(CASE WHEN sh.IS_GOAL = 1 THEN 1 ELSE 0 END) / NULLIF(COUNT(sh.IS_GOAL), 0) * 100, 1) AS FG_PCT_TRACKED,
    ROUND(AVG(sh.DISTANCE_OF_SHOT), 1) AS AVG_SHOT_DISTANCE,
    CURRENT_TIMESTAMP() AS _DATA_INCLUSAO,
    CURRENT_TIMESTAMP() AS _DATA_ALTERACAO
FROM YDS_DB.SILVER.KOBE_SEASONS s
LEFT JOIN YDS_DB.SILVER.SHOTS_ENRICHED sh ON sh.GAME_SEASON = s.SEASON
GROUP BY s.SEASON, s.GAMES, s.PPG, s.WINS, s.LOSSES, s.TOTAL_GAMES_TEAM, s.WIN_PCT;

In [ ]:
%%sql -r create_dim_opponent
-- DIM_OPPONENT: 1 linha = 1 adversário com head-to-head stats
CREATE OR REPLACE DYNAMIC TABLE YDS_DB.GOLD.DIM_OPPONENT
    TARGET_LAG = '1 hour'
    WAREHOUSE = YDS_WH
AS
SELECT
    g.ADVERSARIO_SIGLA AS TEAM_ABBR,
    g.ADVERSARIO_NOME AS TEAM_NAME,
    COUNT(*) AS TOTAL_GAMES,
    SUM(g.IS_WIN) AS WINS_VS,
    SUM(1 - g.IS_WIN) AS LOSSES_VS,
    ROUND(SUM(g.IS_WIN) / NULLIF(COUNT(*), 0) * 100, 1) AS WIN_PCT_VS,
    ROUND(AVG(g.KOBE_POINTS), 1) AS AVG_KOBE_POINTS_VS,
    MAX(g.KOBE_POINTS) AS MAX_KOBE_POINTS_VS,
    SUM(CASE WHEN g.GAME_TYPE = 'PO' THEN 1 ELSE 0 END) AS PLAYOFF_GAMES_VS,
    CURRENT_TIMESTAMP() AS _DATA_INCLUSAO,
    CURRENT_TIMESTAMP() AS _DATA_ALTERACAO
FROM YDS_DB.SILVER.KOBE_GAMELOG g
GROUP BY g.ADVERSARIO_SIGLA, g.ADVERSARIO_NOME;

In [ ]:
%%sql -r create_dim_arena
-- DIM_ARENA: arenas por time com validade temporal (SCD2 natural)
CREATE OR REPLACE DYNAMIC TABLE YDS_DB.GOLD.DIM_ARENA
    TARGET_LAG = '1 hour'
    WAREHOUSE = YDS_WH
AS
SELECT
    TEAM_ABBR,
    TEAM_NAME,
    ARENA_NAME,
    CITY,
    VALID_FROM,
    VALID_TO,
    CURRENT_TIMESTAMP() AS _DATA_INCLUSAO,
    CURRENT_TIMESTAMP() AS _DATA_ALTERACAO
FROM YDS_DB.SILVER.ARENAS;

## Validação

In [ ]:
%%sql -r gold_validation
-- Contagem de todas as tabelas Gold
SELECT 'FCT_GAME' AS TABELA, COUNT(*) AS LINHAS FROM YDS_DB.GOLD.FCT_GAME
UNION ALL SELECT 'FCT_SHOT', COUNT(*) FROM YDS_DB.GOLD.FCT_SHOT
UNION ALL SELECT 'DIM_SEASON', COUNT(*) FROM YDS_DB.GOLD.DIM_SEASON
UNION ALL SELECT 'DIM_OPPONENT', COUNT(*) FROM YDS_DB.GOLD.DIM_OPPONENT
UNION ALL SELECT 'DIM_ARENA', COUNT(*) FROM YDS_DB.GOLD.DIM_ARENA;

In [ ]:
%%sql -r test_81pts
-- Teste: jogo dos 81 pontos completo
SELECT DATE, ADVERSARIO_NOME, ARENA_NAME, CITY, KOBE_POINTS,
    FG2_MADE, FG3_MADE, FG_MADE || '/' || FG_ATT AS FG, FG_PCT || '%' AS FG_PCT,
    FT_MADE || '/' || FT_ATT AS FT, FT_PCT || '%' AS FT_PCT,
    LAL_SCORE || ' x ' || OPP_SCORE AS PLACAR, RESULT
FROM YDS_DB.GOLD.FCT_GAME
WHERE KOBE_POINTS = 81;